# Step 1: Automatic cell segmentation with Cellpose

In [ ]:
import sys
sys.path.append("..")
import os
import tifffile as tiff
from matplotlib import pyplot as plt
from cellpose import models, io
import numpy as np
from utility import config
from utility.data_loader import DataLoader

## User input here
sub_ids = np.array([1,2,3,4,5])
project_dir = config.data_dir

def segment_all_frames(img):
    # Segmentation
    imgs = [img[i,:,:] for i in range(img.shape[0])]
    # Run cellpose
    model = models.CellposeModel(gpu=True)
    masks, flows, styles = model.eval(imgs)
    # Save masks as tif
    masks_all = np.stack(masks, axis=0)
    return masks_all

for sub_id in sub_ids:
    print(f'Processing sub {sub_id}...')
    data_path = os.path.join(project_dir, f"sub-{sub_id:03d}", f"sub-{sub_id:03d}_data-xyProjection.tif")
    membrane_mask_path = os.path.join(project_dir, f"sub-{sub_id:03d}", f"sub-{sub_id:03d}_data-memMask.tif")
    nucleus_mask_path = os.path.join(project_dir, f"sub-{sub_id:03d}", f"sub-{sub_id:03d}_data-nucMask.tif")

    image = tiff.imread(data_path)
    img_ch0 = image[:,0,:,:]
    img_ch1 = image[:,1,:,:]

    masks_all = segment_all_frames(img_ch1)
    tiff.imwrite(nucleus_mask_path, masks_all.astype(np.uint16))
 
    masks_all = segment_all_frames(img_ch0)
    tiff.imwrite(membrane_mask_path, masks_all.astype(np.uint16))

In [ ]:
# Read the tif file
image = tiff.imread(data_path)
# Split the channels
img_ch0 = image[:,0,:,:]
img_ch1 = image[:,1,:,:]
# Visualize one slice of each channel
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_ch0[10,:,:], cmap='gray')
axes[0].set_title('Channel 0')
axes[1].imshow(img_ch1[10,:,:], cmap='gray')
axes[1].set_title('Channel 1')